In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange

## Multi-Armed Bandit with epsilon-greedy and UCB strategies

In [ ]:
class MultiArmedBandit:
    def __init__(self, num_arms):
        self.num_arms = num_arms
        self.q_true = np.random.normal(loc=0, scale=1, size=num_arms)  # True mean rewards of each arm
        self.q_estimates = np.zeros(num_arms)  # Estimated values of each arm
        self.arm_counts = np.zeros(num_arms)  # Number of times each arm has been chosen
        self.total_reward = 0
        self.time_step = 0

    def reset(self):
        self.q_true = np.random.normal(loc=0, scale=1, size=self.num_arms)
        self.q_estimates = np.zeros(self.num_arms)
        self.arm_counts = np.zeros(self.num_arms)
        self.total_reward = 0
        self.time_step = 0

    def epsilon_greedy_action(self, epsilon):
        if np.random.random() < epsilon:
            # Explore: Choose a random arm
            action = np.random.randint(self.num_arms)
        else:
            # Exploit: Choose the arm with the highest estimated value
            action = np.argmax(self.q_estimates)
        return action

    def ucb_action(self, c=1):
        # Upper Confidence Bound action selection
        ucb_estimates = self.q_estimates + c * np.sqrt(np.log(self.time_step + 1) / (self.arm_counts + 1e-5))
        action = np.argmax(ucb_estimates)
        return action

    def step(self, action):
        # Get reward from the chosen arm
        reward = np.random.normal(loc=self.q_true[action], scale=1)
        
        # Update estimates and counts
        self.arm_counts[action] += 1
        self.total_reward += reward
        self.time_step += 1
        self.q_estimates[action] += (reward - self.q_estimates[action]) / self.arm_counts[action]

        return reward

In [ ]:
bandit = MultiArmedBandit(num_arms=10)

In [ ]:
# Number of time steps to run the bandit
num_steps = 1000
# Epsilon value for epsilon-greedy strategy
epsilon2 = 0.2
epsilon1 = 0.1
# UCB parameter
ucb_c = 2
# number of realisations
num_realisations = 1000

In [ ]:
# Arrays to store rewards over time
epsilon_greedy_rewards1 = np.zeros((num_realisations, num_steps))
epsilon_greedy_rewards2 = np.zeros((num_realisations, num_steps))
ucb_rewards = np.zeros((num_realisations, num_steps))

In [ ]:
for ir in trange(num_realisations):
    # Run epsilon-greedy strategy
    bandit.reset()
    for t in range(num_steps):
        action = bandit.epsilon_greedy_action(epsilon1)
        reward = bandit.step(action)
        epsilon_greedy_rewards1[ir, t] = reward

In [ ]:
for ir in trange(num_realisations):
    # Run epsilon-greedy strategy
    bandit.reset()
    for t in range(num_steps):
        action = bandit.epsilon_greedy_action(epsilon2)
        reward = bandit.step(action)
        epsilon_greedy_rewards2[ir, t] = reward

In [ ]:
for ir in trange(num_realisations):
    # Run UCB strategy
    bandit.reset()
    for t in range(num_steps):
        action = bandit.ucb_action(ucb_c)
        reward = bandit.step(action)
        ucb_rewards[ir, t] = reward

In [ ]:
# Calculate cumulative rewards
epsilon_greedy_cumulative_reward1 = np.cumsum(epsilon_greedy_rewards1, axis = 1)
epsilon_greedy_cumulative_reward2 = np.cumsum(epsilon_greedy_rewards2, axis = 1)
ucb_cumulative_reward = np.cumsum(ucb_rewards, axis = 1)

In [ ]:
# average over realisations
ave_egreedy1 = epsilon_greedy_cumulative_reward1.mean(axis=0)
ave_egreedy2 = epsilon_greedy_cumulative_reward2.mean(axis=0)
ave_ucb = ucb_cumulative_reward.mean(axis=0)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ave_egreedy1, 'k-', label='Ave. Epsilon-Greedy $\epsilon = 0.1$')
plt.plot(ave_egreedy2, 'k--', label='Ave. Epsilon-Greedy $\epsilon = 0.2$')
plt.plot(ave_ucb, 'k-.', label='Ave. UCB')
plt.xlabel('Time Steps')
plt.ylabel('Cumulative Reward')
plt.legend()
plt.grid(True)
plt.savefig('multiarmedbandit.png', dpi=300)
plt.show()